In [2]:
import sys
sys.path.append("../")

%load_ext autoreload
%autoreload 2

from src.data.load import load_ratings

ratings = load_ratings()

print("Shape:", ratings.shape)
print(ratings.dtypes)
ratings.head()

Shape: (1048575, 4)
userId         int64
movieId        int64
rating       float64
timestamp      int64
dtype: object


,userId,movieId,rating,timestamp
0,1,296,5.0,1147880044
1,1,306,3.5,1147868817
2,1,307,5.0,1147868828
3,1,665,5.0,1147878820
4,1,899,3.5,1147868510


In [3]:
n_users = ratings["userId"].nunique()
n_movies = ratings["movieId"].nunique()
n_ratings = len(ratings)

possible_cells = n_users * n_movies
sparsity = 1 - (n_ratings / possible_cells)

print(f"Users: {n_users:,}")
print(f"Movies: {n_movies:,}")
print(f"Ratings: {n_ratings:,}")
print(f"Possible cells (dense matrix): {possible_cells:,}")
print(f"Sparsity: {sparsity:.4%}")

Users: 7,045
Movies: 22,240
Ratings: 1,048,575
Possible cells (dense matrix): 156,680,800
Sparsity: 99.3308%


In [4]:
user_ids = ratings["userId"].unique()
movie_ids = ratings["movieId"].unique()

user_id_to_index = {user_id: idx for idx, user_id in enumerate(user_ids)}
movie_id_to_index = {movie_id: idx for idx, movie_id in enumerate(movie_ids)}

index_to_user_id = {idx: user_id for user_id, idx in user_id_to_index.items()}
index_to_movie_id = {idx: movie_id for movie_id, idx in movie_id_to_index.items()}

print("Number of user indices:", len(user_id_to_index))
print("Number of movie indices:", len(movie_id_to_index))

Number of user indices: 7045
Number of movie indices: 22240


In [5]:
from scipy.sparse import csr_matrix
import numpy as np

row_indices = ratings["userId"].map(user_id_to_index).values
col_indices = ratings["movieId"].map(movie_id_to_index).values
rating_values = ratings["rating"].values

user_item_matrix = csr_matrix(
    (rating_values, (row_indices, col_indices)),
    shape=(n_users, n_movies)
)

print("Matrix shape:", user_item_matrix.shape)
print("Matrix dtype:", user_item_matrix.dtype)
print("Non-zero entries:", user_item_matrix.nnz)
print("Matches ratings count:", user_item_matrix.nnz == n_ratings)

Matrix shape: (7045, 22240)
Matrix dtype: float64
Non-zero entries: 1048575
Matches ratings count: True


In [6]:
import joblib
from scipy import sparse
from src.config import (
    USER_ITEM_MATRIX_PATH,
    USER_ID_TO_INDEX_PATH,
    MOVIE_ID_TO_INDEX_PATH,
    INDEX_TO_MOVIE_ID_PATH,
)

sparse.save_npz(USER_ITEM_MATRIX_PATH, user_item_matrix)
joblib.dump(user_id_to_index, USER_ID_TO_INDEX_PATH)
joblib.dump(movie_id_to_index, MOVIE_ID_TO_INDEX_PATH)
joblib.dump(index_to_movie_id, INDEX_TO_MOVIE_ID_PATH)

print("Saved matrix and mappings.")

Saved matrix and mappings.


In [7]:
from sklearn.decomposition import TruncatedSVD

N_COMPONENTS = 50  # number of latent factors — a starting point, we'll tune later

svd = TruncatedSVD(n_components=N_COMPONENTS, random_state=42)

user_factors = svd.fit_transform(user_item_matrix)   # shape: (n_users, N_COMPONENTS)
movie_factors = svd.components_.T                     # shape: (n_movies, N_COMPONENTS)

print("user_factors shape:", user_factors.shape)
print("movie_factors shape:", movie_factors.shape)
print("Explained variance ratio (sum):", svd.explained_variance_ratio_.sum())

user_factors shape: (7045, 50)
movie_factors shape: (22240, 50)
Explained variance ratio (sum): 0.3714462414254448


In [8]:
from sklearn.metrics.pairwise import cosine_similarity

def most_similar_movies_collab(movie_id, n=10):
    if movie_id not in movie_id_to_index:
        return f"movieId {movie_id} not found in matrix."

    idx = movie_id_to_index[movie_id]
    target_vector = movie_factors[idx].reshape(1, -1)

    similarities = cosine_similarity(target_vector, movie_factors).flatten()
    similar_indices = similarities.argsort()[-(n+1):][::-1]
    similar_indices = [i for i in similar_indices if i != idx][:n]

    results = []
    for i in similar_indices:
        results.append({
            "movieId": index_to_movie_id[i],
            "similarity": similarities[i]
        })
    return results

# Toy Story's movieId in MovieLens is 1
most_similar_movies_collab(1, n=10)

[{'movieId': np.int64(3114), 'similarity': np.float64(0.655869482135861)},
 {'movieId': np.int64(979), 'similarity': np.float64(0.5907524632712242)},
 {'movieId': np.int64(1448), 'similarity': np.float64(0.5907524632712242)},
 {'movieId': np.int64(1510), 'similarity': np.float64(0.5907524632712241)},
 {'movieId': np.int64(1524), 'similarity': np.float64(0.5907524632712241)},
 {'movieId': np.int64(1462), 'similarity': np.float64(0.5907524632712241)},
 {'movieId': np.int64(1424), 'similarity': np.float64(0.5907524632712241)},
 {'movieId': np.int64(1470), 'similarity': np.float64(0.5907524632712241)},
 {'movieId': np.int64(1443), 'similarity': np.float64(0.5907524632712241)},
 {'movieId': np.int64(1420), 'similarity': np.float64(0.5907524632712241)}]

In [9]:
from src.data.load import load_movies

movies_lookup = load_movies().set_index("movieId")["title"]

def most_similar_movies_collab_readable(movie_id, n=10):
    results = most_similar_movies_collab(movie_id, n=n)
    if isinstance(results, str):
        return results

    for r in results:
        r["title"] = movies_lookup.get(r["movieId"], "Unknown")

    return results

most_similar_movies_collab_readable(1, n=10)  # Toy Story

[{'movieId': np.int64(3114),
  'similarity': np.float64(0.655869482135861),
  'title': 'Toy Story 2 (1999)'},
 {'movieId': np.int64(979),
  'similarity': np.float64(0.5907524632712242),
  'title': 'Nothing Personal (1995)'},
 {'movieId': np.int64(1448),
  'similarity': np.float64(0.5907524632712242),
  'title': 'Fire on the Mountain (1996)'},
 {'movieId': np.int64(1510),
  'similarity': np.float64(0.5907524632712241),
  'title': "Brother's Kiss, A (1997)"},
 {'movieId': np.int64(1524),
  'similarity': np.float64(0.5907524632712241),
  'title': 'Turning, The (1992)'},
 {'movieId': np.int64(1462),
  'similarity': np.float64(0.5907524632712241),
  'title': 'Unforgotten: Twenty-Five Years After Willowbrook (1996)'},
 {'movieId': np.int64(1424),
  'similarity': np.float64(0.5907524632712241),
  'title': 'Inside (1996)'},
 {'movieId': np.int64(1470),
  'similarity': np.float64(0.5907524632712241),
  'title': 'Rhyme & Reason (1997)'},
 {'movieId': np.int64(1443),
  'similarity': np.float64(0.

In [10]:
from src.recommend import recommend

print("=== Content-based recommendations for Toy Story (1995) ===")
display(recommend("Toy Story (1995)", n=10))

print("\n=== Collaborative recommendations for Toy Story (movieId=1) ===")
for r in most_similar_movies_collab_readable(1, n=10):
    print(f"{r['title']}  (similarity={r['similarity']:.3f})")

=== Content-based recommendations for Toy Story (1995) ===


,movieId,title,genres,similarity_score
3021,3114,Toy Story 2 (1999),Adventure|Animation|Children|Comedy|Fantasy,0.930007
2264,2355,"Bug's Life, A (1998)",Adventure|Animation|Children|Comedy,0.851211
4780,4886,"Monsters, Inc. (2001)",Adventure|Animation|Children|Comedy|Fantasy,0.786317
14813,78499,Toy Story 3 (2010),Adventure|Animation|Children|Comedy|Fantasy|IMAX,0.770890
59767,201588,Toy Story 4 (2019),Adventure|Animation|Children|Comedy,0.730895
39485,157296,Finding Dory (2016),Adventure|Animation|Comedy,0.718785
6258,6377,Finding Nemo (2003),Adventure|Animation|Children|Comedy,0.712220
8246,8961,"Incredibles, The (2004)",Action|Adventure|Animation|Children|Comedy,0.706227
19870,103141,Monsters University (2013),Adventure|Animation|Comedy,0.699954
48035,175831,Lou (2017),Animation,0.673460



=== Collaborative recommendations for Toy Story (movieId=1) ===
Toy Story 2 (1999)  (similarity=0.656)
Nothing Personal (1995)  (similarity=0.591)
Fire on the Mountain (1996)  (similarity=0.591)
Brother's Kiss, A (1997)  (similarity=0.591)
Turning, The (1992)  (similarity=0.591)
Unforgotten: Twenty-Five Years After Willowbrook (1996)  (similarity=0.591)
Inside (1996)  (similarity=0.591)
Rhyme & Reason (1997)  (similarity=0.591)
Tickle in the Heart, A (1996)  (similarity=0.591)
Message to Love: The Isle of Wight Festival (1996)  (similarity=0.591)


In [11]:
import numpy as np

def recommend_for_user(user_id, n=10):
    if user_id not in user_id_to_index:
        return f"userId {user_id} not found."

    user_idx = user_id_to_index[user_id]
    user_vector = user_factors[user_idx]  # shape: (N_COMPONENTS,)

    # Predicted rating for every movie = dot product of user vector and each movie vector
    predicted_scores = movie_factors @ user_vector  # shape: (n_movies,)

    # Exclude movies this user already rated
    already_rated_movie_ids = set(
        ratings.loc[ratings["userId"] == user_id, "movieId"]
    )
    already_rated_indices = {
        movie_id_to_index[mid] for mid in already_rated_movie_ids
        if mid in movie_id_to_index
    }

    # Rank all movie indices by predicted score, skipping already-rated ones
    top_indices = predicted_scores.argsort()[::-1]
    top_indices = [i for i in top_indices if i not in already_rated_indices][:n]

    results = []
    for i in top_indices:
        movie_id = index_to_movie_id[i]
        results.append({
            "movieId": movie_id,
            "title": movies_lookup.get(movie_id, "Unknown"),
            "predicted_score": predicted_scores[i]
        })
    return results

# Try it on an arbitrary real user from your dataset
sample_user_id = ratings["userId"].iloc[0]
print("Recommending for userId:", sample_user_id)
recommend_for_user(sample_user_id, n=10)

Recommending for userId: 1


[{'movieId': np.int64(6874),
  'title': 'Kill Bill: Vol. 1 (2003)',
  'predicted_score': np.float64(1.3985061866490536)},
 {'movieId': np.int64(778),
  'title': 'Trainspotting (1996)',
  'predicted_score': np.float64(1.3212750636019315)},
 {'movieId': np.int64(7153),
  'title': 'Lord of the Rings: The Return of the King, The (2003)',
  'predicted_score': np.float64(1.2335396574659103)},
 {'movieId': np.int64(7438),
  'title': 'Kill Bill: Vol. 2 (2004)',
  'predicted_score': np.float64(1.2195768343833033)},
 {'movieId': np.int64(4993),
  'title': 'Lord of the Rings: The Fellowship of the Ring, The (2001)',
  'predicted_score': np.float64(1.156409652556753)},
 {'movieId': np.int64(4306),
  'title': 'Shrek (2001)',
  'predicted_score': np.float64(1.1462003329311803)},
 {'movieId': np.int64(4226),
  'title': 'Memento (2000)',
  'predicted_score': np.float64(1.1186446250600175)},
 {'movieId': np.int64(4878),
  'title': 'Donnie Darko (2001)',
  'predicted_score': np.float64(1.031460311867668

In [12]:
import numpy as np
import joblib
from src.config import USER_FACTORS_PATH, MOVIE_FACTORS_PATH, SVD_MODEL_PATH

np.save(USER_FACTORS_PATH, user_factors)
np.save(MOVIE_FACTORS_PATH, movie_factors)
joblib.dump(svd, SVD_MODEL_PATH)

print("Saved user_factors, movie_factors, and the fitted SVD model.")

Saved user_factors, movie_factors, and the fitted SVD model.


In [13]:
user_rated_movies = (
    ratings.groupby("userId")["movieId"]
    .apply(set)
    .to_dict()
)

print("Number of users in lookup:", len(user_rated_movies))
print("Example — userId", sample_user_id, "has rated", len(user_rated_movies[sample_user_id]), "movies")

Number of users in lookup: 7045
Example — userId 1 has rated 70 movies


In [14]:
from src.config import USER_RATED_MOVIES_PATH

joblib.dump(user_rated_movies, USER_RATED_MOVIES_PATH)
print("Saved user_rated_movies lookup.")

Saved user_rated_movies lookup.


In [15]:
import sys
sys.path.append("../")

%load_ext autoreload
%autoreload 2

from src.recommend_user import recommend_for_user as recommend_user_standalone

recommend_user_standalone(sample_user_id, n=10)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


[{'movieId': np.int64(6874),
  'predicted_score': np.float64(1.3985061866490536),
  'title': 'Kill Bill: Vol. 1 (2003)'},
 {'movieId': np.int64(778),
  'predicted_score': np.float64(1.3212750636019315),
  'title': 'Trainspotting (1996)'},
 {'movieId': np.int64(7153),
  'predicted_score': np.float64(1.2335396574659103),
  'title': 'Lord of the Rings: The Return of the King, The (2003)'},
 {'movieId': np.int64(7438),
  'predicted_score': np.float64(1.2195768343833033),
  'title': 'Kill Bill: Vol. 2 (2004)'},
 {'movieId': np.int64(4993),
  'predicted_score': np.float64(1.156409652556753),
  'title': 'Lord of the Rings: The Fellowship of the Ring, The (2001)'},
 {'movieId': np.int64(4306),
  'predicted_score': np.float64(1.1462003329311803),
  'title': 'Shrek (2001)'},
 {'movieId': np.int64(4226),
  'predicted_score': np.float64(1.1186446250600175),
  'title': 'Memento (2000)'},
 {'movieId': np.int64(4878),
  'predicted_score': np.float64(1.0314603118676684),
  'title': 'Donnie Darko (2001